In [ ]:
import requests
import pandas as pd
import time
from google.colab import files

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
full_solar_parameters = "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,PRECTOTCORR,T2M"

# 1. Khai báo danh sách các khu vực bạn muốn lấy (Ví dụ tọa độ 3 cơ sở của La Trobe)
# Sau này bạn có thể đọc trực tiếp từ file Solar_Site_Details.csv ra danh sách này
locations = [
    {"site_name": "Bundoora_Campus", "lat": -37.7214, "lon": 145.0483},
    {"site_name": "Bendigo_Campus", "lat": -36.7813, "lon": 144.2982},
    {"site_name": "Albury_Wodonga", "lat": -36.1082, "lon": 146.8923}
]

all_regional_data = [] # Danh sách chứa DataFrame của từng vùng

# 2. Vòng lặp quét qua từng khu vực
for loc in locations:
    print(f"Đang lấy dữ liệu cho: {loc['site_name']}...")

    query_params = {
        "start": 20200101,
        "end": 20220430,
        "latitude": loc["lat"],
        "longitude": loc["lon"],
        "community": "re",
        "parameters": full_solar_parameters,
        "format": "json",
        "units": "metric",
        "time-standard": "lst"
    }

    response = requests.get(url, params=query_params)

    if response.status_code == 200:
        res_data = response.json()
        ts_data = res_data["properties"]["parameter"]

        # Chuyển thành DataFrame
        df_loc = pd.DataFrame(ts_data)
        df_loc.index.name = "Date"
        df_loc.reset_index(inplace=True)

        # Thêm cột tên khu vực để sau này phân biệt khi train model
        df_loc["Site_Name"] = loc["site_name"]
        df_loc["Latitude"] = loc["lat"]
        df_loc["Longitude"] = loc["lon"]

        all_regional_data.append(df_loc)
    else:
        print(f"Lỗi tại trạm {loc['site_name']}: {response.status_code}")

    # Nghỉ 2 giây để tránh bị NASA kích hoạt chặn do spam request
    time.sleep(2)

# 3. Gộp tất cả các khu vực thành một DataFrame tổng siêu khủng
if all_regional_data:
    df_final = pd.concat(all_regional_data, ignore_index=True)
    df_final["Date"] = pd.to_datetime(df_final["Date"], format="%Y%m%d")

    # Đẩy cột Site_Name lên đầu cho dễ nhìn
    cols = ['Date', 'Site_Name', 'Latitude', 'Longitude'] + [c for c in df_final.columns if c not in ['Date', 'Site_Name', 'Latitude', 'Longitude']]
    df_final = df_final[cols]

    print("\n--- ĐÃ GỘP THÀNH CÔNG DỮ LIỆU TẤT CẢ CÁC KHU VỰC ---")
    print(df_final.head(10))

    # Xuất file CSV tổng
    output_file = "NASA_Multi_Zone_Weather.csv"
    df_final.to_csv(output_file, index=False)
    files.download(output_file)